# 07 · Auditoría, linaje y decisión de publicación

**Objetivo:** mostrar qué evidencia se exige antes de habilitar datos para análisis.

**Modalidad:** material para explicar en clase. Los ejemplos se ejecutan únicamente en memoria, con datos ficticios. No conectan a una base, no crean archivos ni ejecutan una ingesta. Los bloques SQL son ilustrativos y no se envían a ningún servidor.

**Prerrequisitos:** Python con pandas. Cada notebook es independiente: no requiere scripts `.py` ni haber ejecutado otra notebook.


In [ ]:
# Importar conexión centralizada desde el módulo
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, SERVIDOR, BD_ORIGEN, BD_DESTINO, USUARIO

# Obtener los engines
engine_origen, engine_destino = obtener_engines()
print(f'Conexiones establecidas correctamente.')
print(f'Servidor: {SERVIDOR}')
print(f'Origen: {BD_ORIGEN}')
print(f'Destino: {BD_DESTINO}')


## Publicar no es imprimir un mensaje de éxito

En una implementación real, dimensiones, hechos, linaje y marcador de éxito
deberían coordinarse en una transacción. Si falla un control crítico, se hace
rollback y el marcador no avanza.

```sql
-- Esquema conceptual: no se ejecuta.
BEGIN TRANSACTION;
-- Aplicar dimensiones y hechos.
-- Verificar grano, FK, SCD e importes.
-- Guardar linaje y marcador de publicación.
COMMIT; -- Solo si los controles y las operaciones fueron correctos.
-- En caso de error: ROLLBACK y registro del fallo.
```


In [ ]:
from decimal import Decimal
import pandas as pd

balance = pd.DataFrame([
    {"tabla": "Sales.InvoiceLines", "extraidas": 3, "aceptadas": 2, "cuarentena": 1},
    {"tabla": "Sales.Invoices", "extraidas": 2, "aceptadas": 2, "cuarentena": 0},
])
balance["reconcilia"] = balance["extraidas"] == balance["aceptadas"] + balance["cuarentena"]
display(balance)

origen_aceptado = {"neto": Decimal("15.00"), "impuesto": Decimal("3.15"), "total": Decimal("18.15")}
destino_propuesto = dict(origen_aceptado)
controles = {
    "balance_filas": bool(balance["reconcilia"].all()),
    "importes": origen_aceptado == destino_propuesto,
    "grano_unico": True,  # En un sistema real: calcular sobre los hechos.
    "referencias_validas": True,  # Aquí son condiciones ilustrativas declaradas.
    "criticos_pendientes": int(balance["cuarentena"].sum()) == 0,
}
display(pd.DataFrame(controles.items(), columns=["control", "cumple"]))
print("Decisión con tolerancia cero:", "PUBLICAR" if all(controles.values()) else "BLOQUEAR")


## Interpretar correctamente el balance

El balance puede cuadrar y aun así **bloquearse** la publicación: una fila
está en cuarentena y el contrato tiene tolerancia cero. Reconciliar no equivale
a aprobar calidad. Las FK y el grano no se verifican aquí: se muestran como
condiciones del caso, no como garantías obtenidas de una base real.


In [ ]:
linaje = pd.DataFrame([
    {"destino": "fact_ventas.neto",
     "origen": "Sales.InvoiceLines.ExtendedPrice, TaxAmount",
     "transformacion": "ExtendedPrice - TaxAmount",
     "lote": "CLASE-001", "nk_detalle": 1},
    {"destino": "fact_ventas.cliente_sk",
     "origen": "Sales.Invoices.CustomerID -> dim_cliente",
     "transformacion": "Búsqueda NK y vigencia acordada",
     "lote": "CLASE-001", "nk_detalle": 1},
])
display(linaje)


## Actividad de discusión

¿Por qué reconciliar contra el origen aceptado y también informar lo rechazado? **Respuesta:** comparar solo contra el subconjunto aceptado podría ocultar el impacto de las exclusiones sobre el total del negocio.